# Transfer Learning - NIH Chest X-Ray Classification

## 1. Setup

In [ ]:
import json
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from tensorflow.keras.applications import ResNet50, DenseNet121, EfficientNetB3
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

warnings.filterwarnings('ignore')
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPUs available: {len(tf.config.list_physical_devices('GPU'))}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!nvidia-smi

## 2. Configuration

In [ ]:
PROJECT_ROOT = Path('/content')
DRIVE_DATA = Path('/content/drive/MyDrive/capstone_data')
PROCESSED_DIR = DRIVE_DATA / 'nih-chest-xray-splits'
MODELS_DIR = PROJECT_ROOT / 'models'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'

MODELS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data directory: {PROCESSED_DIR}")
print(f"Models directory: {MODELS_DIR}")

In [ ]:
CONFIG = {
    'img_height': 224,
    'img_width': 224,
    'channels': 3,
    'batch_size': 64,
    'epochs_stage1': 5,
    'epochs_stage2': 10,
    'learning_rate_stage1': 0.001,
    'learning_rate_stage2': 0.0001,
    'unfreeze_layers': 20,
    'dense_units': 512,
    'dropout_rate': 0.5,
    'early_stopping_patience': 10,
    'reduce_lr_patience': 5,
    'num_classes': 14,
    'use_sample': True,
    'sample_size': 1000,
    'random_state': 42
}

MODELS_TO_TRAIN = ['resnet50', 'densenet121', 'efficientnetb3']

print("Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

## 3. Load Data

In [ ]:
train_df = pd.read_csv(PROCESSED_DIR / 'train_split.csv')
val_df = pd.read_csv(PROCESSED_DIR / 'val_split.csv')
test_df = pd.read_csv(PROCESSED_DIR / 'test_split.csv')

with open(PROCESSED_DIR / 'preprocessing_config.json', 'r') as f:
    prep_config = json.load(f)

disease_classes = prep_config['disease_classes']

print(f"Train: {len(train_df):,} images")
print(f"Val:   {len(val_df):,} images")
print(f"Test:  {len(test_df):,} images")
print(f"Disease classes ({len(disease_classes)}): {disease_classes}")

In [ ]:
if CONFIG['use_sample']:
    sample_size = CONFIG['sample_size']
    train_df = train_df.sample(n=min(sample_size, len(train_df)), random_state=42)
    val_df = val_df.sample(n=min(sample_size // 5, len(val_df)), random_state=42)
    test_df = test_df.sample(n=min(sample_size // 5, len(test_df)), random_state=42)
    print(f"\nUsing sample mode:")
    print(f"  Train: {len(train_df):,} images")
    print(f"  Val:   {len(val_df):,} images")
    print(f"  Test:  {len(test_df):,} images")

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

## 4. Data Generators

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_dataframe(
    train_df,
    x_col='full_path',
    y_col=disease_classes,
    target_size=(CONFIG['img_height'], CONFIG['img_width']),
    batch_size=CONFIG['batch_size'],
    class_mode='raw',
    shuffle=True
)

val_gen = val_datagen.flow_from_dataframe(
    val_df,
    x_col='full_path',
    y_col=disease_classes,
    target_size=(CONFIG['img_height'], CONFIG['img_width']),
    batch_size=CONFIG['batch_size'],
    class_mode='raw',
    shuffle=False
)

print(f"Train batches: {len(train_gen)}")
print(f"Val batches: {len(val_gen)}")

## 5. Model Building

In [ ]:
def build_transfer_model(base_model_class, model_name, config):
    input_shape = (config['img_height'], config['img_width'], config['channels'])
    
    base_model = base_model_class(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )
    
    base_model.trainable = False
    
    inputs = keras.Input(shape=input_shape)
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(config['dense_units'], activation='relu')(x)
    x = layers.Dropout(config['dropout_rate'])(x)
    outputs = layers.Dense(config['num_classes'], activation='sigmoid')(x)
    
    model = keras.Model(inputs, outputs, name=model_name)
    
    return model, base_model

## 6. Training Function

In [ ]:
def train_model(model_class, model_name, train_gen, val_gen, config):
    if model_name.lower() not in [m.lower() for m in MODELS_TO_TRAIN]:
        print(f"\nSkipping {model_name}")
        return None, None
    
    print(f"\n{'='*60}")
    print(f"TRAINING: {model_name}")
    print(f"{'='*60}")
    
    model, base_model = build_transfer_model(model_class, f"{model_name.lower()}_transfer", config)
    
    print(f"\nArchitecture:")
    print(f"  Base layers: {len(base_model.layers)} (frozen)")
    print(f"  Total parameters: {model.count_params():,}")
    
    print(f"\n{'-'*60}")
    print("STAGE 1: Feature Extraction")
    print(f"{'-'*60}")
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=config['learning_rate_stage1']),
        loss='binary_crossentropy',
        metrics=['accuracy', keras.metrics.AUC(name='auc', multi_label=True)]
    )
    
    history_s1 = model.fit(
        train_gen,
        epochs=config['epochs_stage1'],
        validation_data=val_gen,
        callbacks=[
            callbacks.EarlyStopping(
                monitor='val_auc',
                mode='max',
                patience=config['early_stopping_patience'] // 2,
                restore_best_weights=True,
                verbose=0
            ),
            callbacks.ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.5,
                patience=config['reduce_lr_patience'] // 2,
                verbose=0
            )
        ],
        verbose=2
    )
    
    print(f"\n{'-'*60}")
    print("STAGE 2: Fine-Tuning")
    print(f"{'-'*60}")
    
    base_model.trainable = True
    for layer in base_model.layers[:-config['unfreeze_layers']]:
        layer.trainable = False
    
    trainable_layers = sum([1 for layer in base_model.layers if layer.trainable])
    print(f"Unfrozen layers: {trainable_layers} / {len(base_model.layers)}")
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=config['learning_rate_stage2']),
        loss='binary_crossentropy',
        metrics=['accuracy', keras.metrics.AUC(name='auc', multi_label=True)]
    )
    
    model_save_path = MODELS_DIR / f"{model_name.lower()}_transfer_best.keras"
    
    history_s2 = model.fit(
        train_gen,
        epochs=config['epochs_stage2'],
        validation_data=val_gen,
        callbacks=[
            callbacks.ModelCheckpoint(
                str(model_save_path),
                monitor='val_auc',
                mode='max',
                save_best_only=True,
                verbose=0
            ),
            callbacks.EarlyStopping(
                monitor='val_auc',
                mode='max',
                patience=config['early_stopping_patience'],
                restore_best_weights=True,
                verbose=0
            ),
            callbacks.ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.5,
                patience=config['reduce_lr_patience'],
                verbose=0
            )
        ],
        verbose=2
    )
    
    print(f"\nModel saved: {model_save_path}")
    
    return model, {'stage1': history_s1.history, 'stage2': history_s2.history}

## 7. Train Models

In [ ]:
trained_models = {}
training_histories = {}

for model_name, model_class in [
    ('ResNet50', ResNet50),
    ('DenseNet121', DenseNet121),
    ('EfficientNetB3', EfficientNetB3)
]:
    model, history = train_model(
        model_class=model_class,
        model_name=model_name,
        train_gen=train_gen,
        val_gen=val_gen,
        config=CONFIG
    )
    
    if model is not None:
        trained_models[model_name] = model
        training_histories[model_name] = history

print(f"\n{'='*60}")
print("TRAINING COMPLETE")
print(f"{'='*60}")
print(f"\nTrained {len(trained_models)} models:")
for name in trained_models.keys():
    print(f"  ✓ {name}")

## 8. Visualize Training

In [ ]:
for model_name, history in training_histories.items():
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f'{model_name} Training History', fontsize=16)
    
    s1_epochs = len(history['stage1']['loss'])
    s2_epochs = len(history['stage2']['loss'])
    total_epochs = s1_epochs + s2_epochs
    
    epochs_s1 = list(range(1, s1_epochs + 1))
    epochs_s2 = list(range(s1_epochs + 1, total_epochs + 1))
    
    for idx, metric in enumerate(['loss', 'accuracy', 'auc']):
        ax = axes[idx]
        
        train_s1 = history['stage1'][metric]
        val_s1 = history['stage1'][f'val_{metric}']
        train_s2 = history['stage2'][metric]
        val_s2 = history['stage2'][f'val_{metric}']
        
        ax.plot(epochs_s1, train_s1, 'b-', label='Train S1', alpha=0.7)
        ax.plot(epochs_s1, val_s1, 'b--', label='Val S1', alpha=0.7)
        ax.plot(epochs_s2, train_s2, 'r-', label='Train S2', alpha=0.7)
        ax.plot(epochs_s2, val_s2, 'r--', label='Val S2', alpha=0.7)
        
        ax.axvline(x=s1_epochs, color='gray', linestyle=':', alpha=0.5)
        ax.set_xlabel('Epoch')
        ax.set_ylabel(metric.capitalize())
        ax.set_title(metric.capitalize())
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 9. Evaluate Models

In [ ]:
test_datagen = ImageDataGenerator(rescale=1./255)

test_gen = test_datagen.flow_from_dataframe(
    test_df,
    x_col='full_path',
    y_col=disease_classes,
    target_size=(CONFIG['img_height'], CONFIG['img_width']),
    batch_size=CONFIG['batch_size'],
    class_mode='raw',
    shuffle=False
)

results = {}

for model_name, model in trained_models.items():
    print(f"\nEvaluating {model_name}...")
    
    test_loss, test_acc, test_auc = model.evaluate(test_gen, verbose=0)
    
    results[model_name] = {
        'loss': test_loss,
        'accuracy': test_acc,
        'auc': test_auc
    }
    
    print(f"  Loss: {test_loss:.4f}")
    print(f"  Accuracy: {test_acc:.4f}")
    print(f"  AUC: {test_auc:.4f}")

print(f"\n{'='*60}")
print("EVALUATION COMPLETE")
print(f"{'='*60}")

## 10. Compare Models

In [ ]:
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values('auc', ascending=False)

print("\nModel Comparison:")
print(results_df.to_string())

fig, ax = plt.subplots(figsize=(10, 6))
results_df[['accuracy', 'auc']].plot(kind='bar', ax=ax)
ax.set_title('Model Performance Comparison')
ax.set_xlabel('Model')
ax.set_ylabel('Score')
ax.set_ylim(0, 1)
ax.legend(['Accuracy', 'AUC'])
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()